In [ ]:
import os
import tempfile
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns

from recommenders.evaluation.python_evaluation import (
    map_at_k, ndcg_at_k, precision_at_k, recall_at_k,
)
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.models.ncf.ncf_singlenode import NCF as MSRecNCF
from recommenders.utils.constants import (
    DEFAULT_ITEM_COL, DEFAULT_RATING_COL,
    DEFAULT_TIMESTAMP_COL, DEFAULT_USER_COL,
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
np.random.seed(42)

LAYER_CONFIGS = {
    'small':  [64, 32],
    'medium': [128, 64],
    'large':  [256, 128, 64],
    'xlarge': [256, 128, 64, 32],
}
K_VALUES      = [5, 10, 20]
SEARCH_K      = 10
EVAL_USERS    = 500
SEARCH_EPOCHS = 10
FINAL_EPOCHS  = 20

print('Imports OK')


In [2]:
df = pd.read_csv('../data/children_products/clildren_product_cleaned.csv')

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

MIN_INTERACTIONS = 3
user_counts = df_filtered.groupby('Телефон_new').size()
item_counts = df_filtered.groupby('ID_SKU').size()
df_filtered = df_filtered[
    df_filtered['Телефон_new'].isin(user_counts[user_counts >= MIN_INTERACTIONS].index) &
    df_filtered['ID_SKU'].isin(item_counts[item_counts >= MIN_INTERACTIONS].index)
]

print(f'Пользователей: {df_filtered["Телефон_new"].nunique():,}')
print(f'Товаров:       {df_filtered["ID_SKU"].nunique():,}')
print(f'Взаимодействий:{len(df_filtered):,}')

Пользователей: 35,395
Товаров:       22,198
Взаимодействий:314,493


In [3]:
interactions = (
    df_filtered
    .groupby(['Телефон_new', 'ID_SKU'])
    .agg(timestamp=('Дата', 'max'), mean_date=('Дата', 'mean'))
    .reset_index()
)
interactions[DEFAULT_RATING_COL] = 1.0
interactions.rename(columns={
    'Телефон_new': DEFAULT_USER_COL,
    'ID_SKU':      DEFAULT_ITEM_COL,
    'timestamp':   DEFAULT_TIMESTAMP_COL,
}, inplace=True)

interactions = interactions.sort_values(DEFAULT_TIMESTAMP_COL)
split_ts = interactions[DEFAULT_TIMESTAMP_COL].quantile(0.7)
split_date = pd.Timestamp(split_ts)
print(f'Дата разделения: {split_date}')

train_df = interactions[interactions[DEFAULT_TIMESTAMP_COL] <  split_ts].copy()
test_df  = interactions[interactions[DEFAULT_TIMESTAMP_COL] >= split_ts].copy()

train_users = set(train_df[DEFAULT_USER_COL].unique())
test_df = test_df[test_df[DEFAULT_USER_COL].isin(train_users)]

print(f'Train: {len(train_df):,} пар, {train_df[DEFAULT_USER_COL].nunique():,} users')
print(f'Test:  {len(test_df):,} пар,  {test_df[DEFAULT_USER_COL].nunique():,} users')

Дата разделения: 2017-04-12 17:37:00
Train: 201,147 пар, 28,017 users
Test:  42,415 пар,  7,128 users


In [ ]:
user2id = {u: i for i, u in enumerate(interactions[DEFAULT_USER_COL].unique())}
item2id = {it: i for i, it in enumerate(interactions[DEFAULT_ITEM_COL].unique())}

def encode_ms(df, u2id=None, i2id=None):
    if u2id is None: u2id = user2id
    if i2id is None: i2id = item2id
    d = df.copy()
    d[DEFAULT_USER_COL] = d[DEFAULT_USER_COL].map(u2id)
    d[DEFAULT_ITEM_COL] = d[DEFAULT_ITEM_COL].map(i2id)
    return d.dropna(subset=[DEFAULT_USER_COL, DEFAULT_ITEM_COL])

train_enc = encode_ms(train_df).sort_values(DEFAULT_USER_COL)
test_enc  = encode_ms(test_df).sort_values(DEFAULT_USER_COL)

tmp_dir    = tempfile.mkdtemp()
train_file = os.path.join(tmp_dir, 'train.csv')
test_file  = os.path.join(tmp_dir, 'test.csv')

train_enc[[DEFAULT_USER_COL, DEFAULT_ITEM_COL, DEFAULT_RATING_COL]].to_csv(train_file, index=False)
test_enc[[DEFAULT_USER_COL, DEFAULT_ITEM_COL, DEFAULT_RATING_COL]].to_csv(test_file,  index=False)

data = NCFDataset(
    train_file=train_file, test_file=test_file, seed=42, n_neg=4, n_neg_test=99,
    col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
)
print(f'NCFDataset: {data.n_users:,} users, {data.n_items:,} items')

INFO:recommenders.models.ncf.dataset:Indexing /var/folders/2s/l9p15g5j1t3gkzyssvhh423r0000gn/T/tmpljygebrl/train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing /var/folders/2s/l9p15g5j1t3gkzyssvhh423r0000gn/T/tmpljygebrl/test.csv ...
INFO:recommenders.models.ncf.dataset:Creating full leave-one-out test file /var/folders/2s/l9p15g5j1t3gkzyssvhh423r0000gn/T/tmpljygebrl/test_full.csv ...
100%|██████████| 7128/7128 [00:23<00:00, 306.09it/s]
INFO:recommenders.models.ncf.dataset:Indexing /var/folders/2s/l9p15g5j1t3gkzyssvhh423r0000gn/T/tmpljygebrl/test_full.csv ...


NCFDataset: 28,017 users, 20,699 items


In [8]:
rng = np.random.default_rng(42)
test_users_eval = [
    u for u in test_enc[DEFAULT_USER_COL].unique()
    if u in set(user2id.values())
]
eval_sample = rng.choice(
    test_users_eval,
    size=min(EVAL_USERS, len(test_users_eval)),
    replace=False
).tolist()


def quick_eval_ms(model, eval_users, test_enc, k=SEARCH_K):
    known_users = set(model.user2id.keys())
    known_items = sorted(model.item2id.keys())
    users = [u for u in eval_users if u in known_users]
    preds = []
    for u in users:
        scores = model.predict([u] * len(known_items), known_items, is_list=True)
        for item, score in zip(known_items, scores):
            preds.append({DEFAULT_USER_COL: u, DEFAULT_ITEM_COL: item, DEFAULT_RATING_COL: score})
    pred_df = pd.DataFrame(preds)
    test_sub = test_enc[
        test_enc[DEFAULT_USER_COL].isin(set(users)) &
        test_enc[DEFAULT_ITEM_COL].isin(set(known_items))
    ]
    return ndcg_at_k(
        test_sub, pred_df, k=k,
        col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
        col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL,
    )


def full_eval_ms(model, test_enc, k_values=K_VALUES):
    known_users = set(model.user2id.keys())
    known_items = sorted(model.item2id.keys())
    test_users  = [u for u in test_enc[DEFAULT_USER_COL].unique() if u in known_users]
    preds = []
    for u in test_users:
        scores = model.predict([u] * len(known_items), known_items, is_list=True)
        for item, score in zip(known_items, scores):
            preds.append({DEFAULT_USER_COL: u, DEFAULT_ITEM_COL: item, DEFAULT_RATING_COL: score})
    pred_df = pd.DataFrame(preds)
    test_sub = test_enc[
        test_enc[DEFAULT_USER_COL].isin(set(test_users)) &
        test_enc[DEFAULT_ITEM_COL].isin(set(known_items))
    ]
    results = {}
    for k in k_values:
        results[k] = {
            'precision': precision_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
            'recall':    recall_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
            'map':       map_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
            'ndcg':      ndcg_at_k(test_sub, pred_df, k=k,
                col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL,
                col_rating=DEFAULT_RATING_COL, col_prediction=DEFAULT_RATING_COL),
        }
    return results


print(f'Quick-eval sample: {len(eval_sample)} users')

Quick-eval sample: 500 users


In [ ]:
t0 = time.time()
baseline_model = MSRecNCF(
    n_users=data.n_users, n_items=data.n_items,
    model_type='NeuMF', n_factors=64,
    layer_sizes=[256, 128, 64], n_epochs=FINAL_EPOCHS,
    batch_size=1024, learning_rate=1e-3, verbose=2, seed=42,
)
baseline_model.fit(data)
print(f'Готово за {time.time() - t0:.0f}s')

Обучение baseline NCF (NeuMF, 20 эпох)...


2026-04-30 00:08:16.654768: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-30 00:08:16.654789: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 2 [182.72s]: train_loss = 3.960943 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 4 [166.34s]: train_loss = 3.757788 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 6 [181.52s]: train_loss = 4.090167 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 8 [181.97s]: train_loss = 5.613730 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [191.23s]: train_loss = 5.828045 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 12 [17

Готово за 3612s


In [ ]:
baseline_results = full_eval_ms(baseline_model, test_enc)
for k in K_VALUES:
    r = baseline_results[k]
    print(f'  K={k}: P={r["precision"]:.4f}  R={r["recall"]:.4f}  MAP={r["map"]:.4f}  NDCG={r["ndcg"]:.4f}')

In [ ]:
def make_ncf_objective(train_file, test_file, eval_users, test_enc, n_epochs=SEARCH_EPOCHS):
    def objective(trial):
        n_factors = trial.suggest_int('n_factors', 16, 128)
        lr        = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        layer_key = trial.suggest_categorical('layers', list(LAYER_CONFIGS.keys()))
        n_neg     = trial.suggest_int('n_neg', 2, 8)

        data_t = NCFDataset(
            train_file=train_file, test_file=test_file, seed=42,
            n_neg=n_neg, n_neg_test=99,
            col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
        )
        m = MSRecNCF(
            n_users=data_t.n_users, n_items=data_t.n_items,
            model_type='NeuMF', n_factors=n_factors,
            layer_sizes=LAYER_CONFIGS[layer_key], n_epochs=n_epochs,
            batch_size=1024, learning_rate=lr, verbose=0, seed=42,
        )
        m.fit(data_t)
        return quick_eval_ms(m, eval_users, test_enc, k=SEARCH_K)
    return objective


N_TRIALS = 40
print(f'Запуск Optuna: {N_TRIALS} trials, {SEARCH_EPOCHS} эпох каждый...')
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    make_ncf_objective(train_file, test_file, eval_sample, test_enc),
    n_trials=N_TRIALS, show_progress_bar=True,
)
print(f'\nЛучший NDCG@{SEARCH_K}: {study.best_value:.4f}')
print(f'Лучшие параметры: {study.best_params}')

In [ ]:
trials_df = study.trials_dataframe()
top10 = trials_df.sort_values('value', ascending=False).head(10)
display(top10[['number', 'value', 'params_n_factors', 'params_lr', 'params_layers', 'params_n_neg']]
        .rename(columns={'value': f'NDCG@{SEARCH_K}'}))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trials_df['number'], trials_df['value'], alpha=0.5, linewidth=1)
ax.plot(trials_df['number'],
        trials_df['value'].cummax(), linewidth=2, color='red', label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel(f'NDCG@{SEARCH_K}')
ax.set_title('Optuna: прогресс поиска гиперпараметров')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
bp = study.best_params
BEST_N_FACTORS = bp['n_factors']
BEST_LR        = bp['lr']
BEST_LAYERS    = LAYER_CONFIGS[bp['layers']]
BEST_N_NEG     = bp['n_neg']
print(f'Лучшая конфигурация: n_factors={BEST_N_FACTORS}, lr={BEST_LR:.2e}, layers={BEST_LAYERS}, n_neg={BEST_N_NEG}')

data_best = NCFDataset(
    train_file=train_file, test_file=test_file, seed=42,
    n_neg=BEST_N_NEG, n_neg_test=99,
    col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
)
t0 = time.time()
best_ms_model = MSRecNCF(
    n_users=data_best.n_users, n_items=data_best.n_items,
    model_type='NeuMF', n_factors=BEST_N_FACTORS,
    layer_sizes=BEST_LAYERS, n_epochs=FINAL_EPOCHS,
    batch_size=1024, learning_rate=BEST_LR, verbose=2, seed=42,
)
best_ms_model.fit(data_best)
print(f'Готово за {time.time() - t0:.0f}s')

best_ms_results = full_eval_ms(best_ms_model, test_enc)
print(f'\n{"K":>4}  {"Метрика":>12}  {"Baseline":>9}  {"BestHP":>9}  {"Δ":>8}')
print('-' * 52)
for k in K_VALUES:
    for m in ['precision', 'recall', 'map', 'ndcg']:
        base = baseline_results[k][m]
        best = best_ms_results[k][m]
        d = best - base
        print(f'{k:>4}  {m+"@"+str(k):>12}  {base:>9.4f}  {best:>9.4f}  {"+" if d>=0 else ""}{d:>7.4f}')

In [ ]:
train_with_time = encode_ms(train_df).copy()

days_before_split = (
    split_date - train_df['mean_date']
).dt.total_seconds().values / 86400
days_before_split = np.clip(days_before_split, 0, None)

DECAY_LAMBDAS = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
decay_scores  = []

for lam in DECAY_LAMBDAS:
    t0 = time.time()
    w = np.exp(-lam * days_before_split).clip(1e-6)
    train_d = train_with_time[[DEFAULT_USER_COL, DEFAULT_ITEM_COL]].copy()
    train_d[DEFAULT_RATING_COL] = w

    decay_file = os.path.join(tmp_dir, f'train_decay_{lam}.csv')
    train_d.to_csv(decay_file, index=False)

    data_d = NCFDataset(
        train_file=decay_file, test_file=test_file, seed=42,
        n_neg=BEST_N_NEG, n_neg_test=99,
        col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
    )
    m_d = MSRecNCF(
        n_users=data_d.n_users, n_items=data_d.n_items,
        model_type='NeuMF', n_factors=BEST_N_FACTORS,
        layer_sizes=BEST_LAYERS, n_epochs=SEARCH_EPOCHS,
        batch_size=1024, learning_rate=BEST_LR, verbose=0, seed=42,
    )
    m_d.fit(data_d)
    score = quick_eval_ms(m_d, eval_sample, test_enc, k=SEARCH_K)
    decay_scores.append(score)
    print(f'  λ={lam:.3f}  NDCG@{SEARCH_K}={score:.4f}  ({time.time()-t0:.0f}s)')

BEST_LAMBDA = DECAY_LAMBDAS[int(np.argmax(decay_scores))]
print(f'\nЛучшая λ = {BEST_LAMBDA}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(DECAY_LAMBDAS, decay_scores, marker='o', linewidth=2)
ax.axvline(BEST_LAMBDA, color='red', linestyle='--', alpha=0.6, label=f'λ*={BEST_LAMBDA}')
ax.set_xscale('log')
ax.set_xlabel('λ (log scale)')
ax.set_ylabel(f'NDCG@{SEARCH_K} (quick eval, {EVAL_USERS} users)')
ax.set_title('Влияние временного затухания на качество NCF')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
w_best = np.exp(-BEST_LAMBDA * days_before_split).clip(1e-6)
train_best_d = train_with_time[[DEFAULT_USER_COL, DEFAULT_ITEM_COL]].copy()
train_best_d[DEFAULT_RATING_COL] = w_best
decay_best_file = os.path.join(tmp_dir, 'train_decay_best.csv')
train_best_d.to_csv(decay_best_file, index=False)

data_decay_best = NCFDataset(
    train_file=decay_best_file, test_file=test_file, seed=42,
    n_neg=BEST_N_NEG, n_neg_test=99,
    col_user=DEFAULT_USER_COL, col_item=DEFAULT_ITEM_COL, col_rating=DEFAULT_RATING_COL,
)
t0 = time.time()
decay_model = MSRecNCF(
    n_users=data_decay_best.n_users, n_items=data_decay_best.n_items,
    model_type='NeuMF', n_factors=BEST_N_FACTORS,
    layer_sizes=BEST_LAYERS, n_epochs=FINAL_EPOCHS,
    batch_size=1024, learning_rate=BEST_LR, verbose=2, seed=42,
)
decay_model.fit(data_decay_best)
print(f'Готово за {time.time() - t0:.0f}s')

decay_results = full_eval_ms(decay_model, test_enc)

In [ ]:
all_variants = {
    'baseline_ms':     baseline_results,
    'best_hparams_ms': best_ms_results,
    'time_decay_ms':   decay_results,
}

ref = baseline_results

print(f'{"Вариант":>22}  {"NDCG@10":>9}  {"MAP@10":>9}  {"ΔNDCG@10":>10}')
print('-' * 60)
for name, res in all_variants.items():
    ndcg = res[10]['ndcg']
    mmap = res[10]['map']
    d = ndcg - ref[10]['ndcg']
    print(f'{name:>22}  {ndcg:>9.4f}  {mmap:>9.4f}  {"+" if d>=0 else ""}{d:>9.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = sns.color_palette('tab10', len(all_variants))

for ax, metric in zip(axes, ['ndcg', 'map']):
    for (name, res), color in zip(all_variants.items(), colors):
        y = [res[k][metric] for k in K_VALUES]
        ax.plot(K_VALUES, y, marker='o', linewidth=1.5, label=name, color=color)
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('NCF Experiments — сравнение вариантов (MS Recommenders)', fontsize=12)
plt.tight_layout()
plt.show()
